In [1]:
# Inline plotting for notebooks
%matplotlib inline

# System and path setup
import sys
import os
from glob import glob
from copy import deepcopy
sys.path.append("../Al_data")

# Standard libraries
import datetime
import random
import pickle as pkl

# Scientific computing
import numpy as np
import pandas as pd
from scipy import stats

# Visualization
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Matminer
from matminer.featurizers.function import FunctionFeaturizer

# Scikit-learn
from sklearn.dummy import DummyRegressor
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import (
    train_test_split, cross_validate, cross_val_predict, 
    RepeatedKFold, GridSearchCV
)
from sklearn.linear_model import BayesianRidge, LinearRegression, Lasso, LassoLars
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, GlobalAveragePooling1D, 
    LayerNormalization, MultiHeadAttention, Embedding, Add, Layer
)

# Project-specific utilities
from stopping_power_ml.utils.io import load_qbox_data

# Keras Finetuning Libraries
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling1D
import kerastuner as kt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Activation, Bidirectional, Normalization
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
import numpy as np
import random

import os
import numpy as np
import pandas as pd
from itertools import product
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import Sequential, Input, regularizers
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Normalization
from tensorflow.keras.callbacks import EarlyStopping


/u/ktrigueiro/.local/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2025-07-15 11:31:22.980089: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-15 11:31:22.984136: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-15 11:31:22.994479: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752597083.012404 1984907 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752597083.017584 1984907 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempti

In [2]:
sys.path.append('./stopping-power-ML-reu/')
from nn_utilities import plot_training_history, plot_loglog_mae, create_datasets, prepare_sequences

In [3]:
data = pd.read_pickle(os.path.join('..', 'data', 'random_data_chrg_1_atoms.pkl.gz'))
print('Data set size:', len(data))
data.head()

Data set size: 9800


,frame_id,force,position,velocity,energy,file_id,file,timestep,displacement,directory,...,AGNI_x in Al eta=6.80e+00,AGNI_y in Al eta=6.80e+00,AGNI_z in Al eta=6.80e+00,AGNI_x in Al eta=1.04e+01,AGNI_y in Al eta=1.04e+01,AGNI_z in Al eta=1.04e+01,AGNI_x in Al eta=1.60e+01,AGNI_y in Al eta=1.60e+01,AGNI_z in Al eta=1.60e+01,initial
14700,0,-0.000129,"[0.0, 5.74196597, 5.74196597]",1.0,-18651.709649,1,datasets/256_Al/Dv1.0,0,0.000000,datasets/256_Al/Dv1.0,...,0.000007,-0.000002,-0.000002,0.000010,-0.000006,-0.000006,0.000011,-0.000007,-0.000007,True
14701,1,0.028183,"[0.00331934, 5.75126463, 5.75228826]",1.0,-18651.701982,1,datasets/256_Al/Dv1.0,1,0.014284,datasets/256_Al/Dv1.0,...,0.000266,-0.003665,-0.003535,0.000266,-0.003546,-0.003424,0.000286,-0.003467,-0.003358,True
14702,2,0.054346,"[0.00663869, 5.76056328, 5.76261055]",1.0,-18651.701392,1,datasets/256_Al/Dv1.0,2,0.028568,datasets/256_Al/Dv1.0,...,0.000525,-0.007327,-0.007067,0.000523,-0.007087,-0.006841,0.000560,-0.006927,-0.006709,True
14703,3,0.080006,"[0.00995803, 5.76986194, 5.77293284]",1.0,-18651.700428,1,datasets/256_Al/Dv1.0,3,0.042852,datasets/256_Al/Dv1.0,...,0.000785,-0.010990,-0.010598,0.000781,-0.010628,-0.010259,0.000834,-0.010389,-0.010061,True
14704,4,0.104996,"[0.01327738, 5.77916059, 5.78325513]",1.0,-18651.699098,1,datasets/256_Al/Dv1.0,4,0.057136,datasets/256_Al/Dv1.0,...,0.001047,-0.014652,-0.014130,0.001040,-0.014169,-0.013677,0.001109,-0.013851,-0.013413,True


In [4]:
data.query('initial == False', inplace=True)
print('Training set size:', len(data))


Training set size: 9379


In [5]:
featurizers = pkl.load(open(os.path.join('..', 'data', 'featurizers_chrg_1_atoms.pkl'), 'rb'))

Configure parameters

In [6]:
X_cols = featurizers.feature_labels()
y_col = 'force'
max_epochs = 70
trials_per_dataset = 15

search_space = {
    'batch_size': [8, 16, 32, 64, 128],
    'timesteps': [5, 10, 20, 25, 30, 40, 50, 70, 100],
    'l1_kernel': [0.0, 0.001, 0.01],
    'l1_recurrent': [0.0, 0.001, 0.01]
}


In [7]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import Sequential, Input, regularizers
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Normalization
from tensorflow.keras.callbacks import EarlyStopping

In [8]:
# ---- Sequence Creation ----
def create_sequences(df, timesteps, X_cols, y_col, group_col='file_id'):
    sequences, targets = [], []
    for _, group in df.groupby(group_col):
        group = group.sort_values('timestep')
        x = group[X_cols].values
        y = group[y_col].values
        for i in range(len(group) - timesteps):
            sequences.append(x[i:i + timesteps])
            targets.append(y[i + timesteps])
    return np.array(sequences), np.array(targets).reshape(-1, 1)

In [9]:
# ---- Model Definition ----
def build_model(input_shape, l1_kernel, l1_recurrent):
    model = Sequential()
    model.add(Input(shape=input_shape))

    norm_layer = Normalization()
    norm_layer.adapt(np.random.random((100, input_shape[-1])))  # Dummy adapt
    model.add(norm_layer)

    model.add(Bidirectional(LSTM(
        32,
        kernel_regularizer=regularizers.L1(l1_kernel),
        recurrent_regularizer=regularizers.L1(l1_recurrent)
    )))
    model.add(Dropout(0.1))
    model.add(Dense(1))  # Regression output
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

In [10]:
# ---- Grid Search ----
def run_grid_search(df, X_cols, y_col, search_space, max_epochs=50):
    all_results = []
    keys = list(search_space.keys())
    combos = list(product(*search_space.values()))

    for combo in combos:
        config = dict(zip(keys, combo))
        print(f"\n🔁 Trying config: {config}")
        try:
            # Prepare data
            X, y, x_scalar, y_scalar = create_sequences(df, config['timesteps'], X_cols, y_col)
            X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=42)

            # Build and train model
            model = build_model((config['timesteps'], len(X_cols)), config['l1_kernel'], config['l1_recurrent'])
            early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

            history = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                batch_size=config['batch_size'],
                epochs=max_epochs,
                callbacks=[early_stop],
                verbose=0
            )

            # Evaluate
            val_mae = min(history.history['val_mae'])
            config['val_mae'] = val_mae
            all_results.append(config)
            print(f"✅ MAE: {val_mae:.4f}")
        except Exception as e:
            print(f"❌ Failed config {config} — {e}")

    best = min(all_results, key=lambda x: x['val_mae'])
    print("\n🏆 Best Configuration:")
    print(best)
    return best, pd.DataFrame(all_results)

In [11]:
import random
from itertools import product

def run_random_search(df, X_cols, y_col, search_space, max_epochs=50, n_trials=10, seed=42):
    all_results = []
    keys = list(search_space.keys())
    combos = list(product(*search_space.values()))
    
    random.seed(seed)
    sampled_combos = random.sample(combos, min(n_trials, len(combos)))

    for combo in sampled_combos:
        config = dict(zip(keys, combo))
        print(f"\n🔁 Trying config: {config}")
        try:
            # Prepare data
            X, y = create_sequences(df, config['timesteps'], X_cols, y_col)
            X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=42)

            # Build and train model
            model = build_model((config['timesteps'], len(X_cols)), config['l1_kernel'], config['l1_recurrent'])
            early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

            history = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                batch_size=config['batch_size'],
                epochs=max_epochs,
                callbacks=[early_stop],
                verbose=0
            )

            # Evaluate
            val_mae = min(history.history['val_mae'])
            config['val_mae'] = val_mae
            all_results.append(config)
            print(f"✅ MAE: {val_mae:.4f}")
        except Exception as e:
            print(f"❌ Failed config {config} — {e}")

    if not all_results:
        raise ValueError("All random search configurations failed.")
        
    best = min(all_results, key=lambda x: x['val_mae'])
    print("\n🏆 Best Configuration:")
    print(best)
    return best, pd.DataFrame(all_results)


In [12]:
best_dataload_config, all_results = run_random_search(data, X_cols, y_col, search_space, max_epochs=70, n_trials=20)
all_results.to_csv("gridsearch_results_single.csv", index=False)


🔁 Trying config: {'batch_size': 128, 'timesteps': 5, 'l1_kernel': 0.001, 'l1_recurrent': 0.0}


2025-07-15 11:31:29.383029: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


✅ MAE: 0.0290

🔁 Trying config: {'batch_size': 8, 'timesteps': 50, 'l1_kernel': 0.001, 'l1_recurrent': 0.0}
✅ MAE: 0.0450

🔁 Trying config: {'batch_size': 8, 'timesteps': 10, 'l1_kernel': 0.001, 'l1_recurrent': 0.0}
✅ MAE: 0.0256

🔁 Trying config: {'batch_size': 128, 'timesteps': 50, 'l1_kernel': 0.0, 'l1_recurrent': 0.001}
✅ MAE: 0.0511

🔁 Trying config: {'batch_size': 16, 'timesteps': 50, 'l1_kernel': 0.001, 'l1_recurrent': 0.01}
✅ MAE: 0.0586

🔁 Trying config: {'batch_size': 16, 'timesteps': 30, 'l1_kernel': 0.01, 'l1_recurrent': 0.01}
✅ MAE: 0.0417

🔁 Trying config: {'batch_size': 16, 'timesteps': 25, 'l1_kernel': 0.01, 'l1_recurrent': 0.0}
✅ MAE: 0.0510

🔁 Trying config: {'batch_size': 8, 'timesteps': 70, 'l1_kernel': 0.01, 'l1_recurrent': 0.01}
✅ MAE: 0.0526

🔁 Trying config: {'batch_size': 128, 'timesteps': 40, 'l1_kernel': 0.01, 'l1_recurrent': 0.01}
✅ MAE: 0.0820

🔁 Trying config: {'batch_size': 8, 'timesteps': 40, 'l1_kernel': 0.01, 'l1_recurrent': 0.001}
✅ MAE: 0.0674

🔁 Try

In [13]:
best_dataload_config, all_results = run_grid_search(data, X_cols, y_col, search_space)
all_results.to_csv("gridsearch_results_single.csv", index=False)


🔁 Trying config: {'batch_size': 8, 'timesteps': 5, 'l1_kernel': 0.0, 'l1_recurrent': 0.0}
❌ Failed config {'batch_size': 8, 'timesteps': 5, 'l1_kernel': 0.0, 'l1_recurrent': 0.0} — not enough values to unpack (expected 4, got 2)

🔁 Trying config: {'batch_size': 8, 'timesteps': 5, 'l1_kernel': 0.0, 'l1_recurrent': 0.001}
❌ Failed config {'batch_size': 8, 'timesteps': 5, 'l1_kernel': 0.0, 'l1_recurrent': 0.001} — not enough values to unpack (expected 4, got 2)

🔁 Trying config: {'batch_size': 8, 'timesteps': 5, 'l1_kernel': 0.0, 'l1_recurrent': 0.01}
❌ Failed config {'batch_size': 8, 'timesteps': 5, 'l1_kernel': 0.0, 'l1_recurrent': 0.01} — not enough values to unpack (expected 4, got 2)

🔁 Trying config: {'batch_size': 8, 'timesteps': 5, 'l1_kernel': 0.001, 'l1_recurrent': 0.0}
❌ Failed config {'batch_size': 8, 'timesteps': 5, 'l1_kernel': 0.001, 'l1_recurrent': 0.0} — not enough values to unpack (expected 4, got 2)

🔁 Trying config: {'batch_size': 8, 'timesteps': 5, 'l1_kernel': 0.001

ValueError: min() arg is an empty sequence

Best config found: 
🔁 Trying config: {'batch_size': 8, 'timesteps': 50, 'l1_kernel': 0.01, 'l1_recurrent': 0.001}
✅ MAE: 0.0520

In [ ]:
timesteps = 10


X, y, x_scaler, y_scaler = prepare_sequences(
                                        data,
                                        X_cols,
                                        y_col,
                                        sequence_col='file_id',
                                        timestep_col='timestep',
                                        timesteps=timesteps,
                                    )

In [ ]:
train_split = 0.5
batch_size = 128

train_ds, val_ds, input_shape = create_datasets(X,
                                                y,
                                                batch_size=batch_size,
                                                train_split=train_split)

In [ ]:
class TransformerEncoder(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=None):
        attn_output = self.att(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)
    
# === Positional Embedding Layer ===
class TimePositionalEmbedding(Layer):
    def __init__(self, sequence_length, embed_dim):
        super().__init__()
        self.pos_embedding = Embedding(input_dim=sequence_length, output_dim=embed_dim)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        pos_encoding = self.pos_embedding(positions)
        return x + pos_encoding
# == Positional Embedding Layer Definition ==
class TimePositionalEmbedding(Layer):
    def __init__(self, sequence_length, embed_dim):
        super().__init__()
        self.pos_embedding = Embedding(input_dim=sequence_length, output_dim=embed_dim)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        pos_encoding = self.pos_embedding(positions)
        return x + pos_encoding
    
# === Updated model builder with positional embedding toggle ===
def build_transformer_model(input_shape, embed_dim=8, num_heads=2, ff_dim=16, dropout_rate=0.2, use_positional_embedding=True):
    inputs = Input(shape=input_shape)
    x = Dense(embed_dim)(inputs)
    if use_positional_embedding:
        x = TimePositionalEmbedding(sequence_length=input_shape[0], embed_dim=embed_dim)(x)
    x = TransformerEncoder(embed_dim=embed_dim, num_heads=num_heads, ff_dim=ff_dim)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1)(x)
    return Model(inputs=inputs, outputs=outputs)

In [ ]:
best_config = {'batch_size': 8, 'timesteps': 50, 'l1_kernel': 0.01, 'l1_recurrent': 0.001}

In [ ]:
input_shape = (X.shape[1], X.shape[2])  # timesteps, features

keras.backend.clear_session()
K.clear_session()
model = build_transformer_model(
            input_shape=input_shape,
            embed_dim=best_config['embed_dim'],
            num_heads=best_config['num_heads'],
            ff_dim=best_config['ff_dim'],
            dropout_rate=best_config['dropout_rate'],
            use_positional_embedding=best_config['use_positional_embedding']
        )
optimizer = tf.keras.optimizers.Adam(learning_rate=best_config['learning_rate'])
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', restore_best_weights=True, patience=30)

history = model.fit(train_ds,
                    epochs=100,
                    validation_data=val_ds,
                    callbacks=[early_stop])


### Create Plotting Functions

In [ ]:
import matplotlib.pyplot as plt
def plot_training_history(history, log_scale=True, style='fivethirtyeight'):
    """
    Plots training and validation loss & MAE from a Keras history object.

    Parameters:
    - history: Keras History object from model.fit()
    - log_scale: Whether to use symlog scale for y-axis (default: True)
    - style: Matplotlib style to use (default: 'fivethirtyeight')
    """
    #plt.style.use(style)

    fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

    # --- Subplot 1: Loss ---
    axs[0].plot(history.history['loss'], label='Training Loss', color='tab:blue', linewidth=2)
    axs[0].plot(history.history['val_loss'], label='Validation Loss', color='tab:orange', linewidth=2)
    axs[0].set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
    axs[0].set_xlabel('Epoch', fontsize=12)
    axs[0].set_ylabel('Loss (MSE)', fontsize=12)
    axs[0].grid(True, which='both', linestyle='--', linewidth=0.5)
    axs[0].legend()
    if log_scale:
        axs[0].set_yscale('symlog')
    axs[0].tick_params(labelsize=10)

    # --- Subplot 2: MAE ---
    axs[1].plot(history.history['mae'], label='Training MAE', color='tab:green', linewidth=2)
    axs[1].plot(history.history['val_mae'], label='Validation MAE', color='tab:red', linewidth=2)
    axs[1].set_title('Training vs Validation MAE', fontsize=14, fontweight='bold')
    axs[1].set_xlabel('Epoch', fontsize=12)
    axs[1].set_ylabel('MAE', fontsize=12)
    axs[1].grid(True, which='both', linestyle='--', linewidth=0.5)
    axs[1].legend()
    if log_scale:
        axs[1].set_yscale('symlog')
    axs[1].tick_params(labelsize=10)

    plt.suptitle('Model Training Performance', fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
def plot_loglog_mae(history, figsize=(3.5, 2.5), line_color='black'):
    plt.figure(figsize=figsize)
    plt.plot(history.history['mae'], label='Dense', color=line_color, linewidth=1.5)

    plt.xscale('log')
    plt.yscale('log')

    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel(r'MAE ($E_h / a_B$)', fontsize=10)

    plt.legend(frameon=False, loc='upper right')

    plt.tick_params(axis='both', which='major', labelsize=8)
    plt.grid(False)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_training_history(history, log_scale=True)

### Test model on full dataset

In [ ]:
# Load new data — it should look like your original df
new_file_df = data[data['file'] == 'datasets/256_Al/Dv1.0']  # or whatever new file

# Make sure it's sorted by timestep
new_file_df = new_file_df.sort_values('timestep')
new_file_df

In [ ]:
pred_sequences = []

for i in range(len(new_file_df) - timesteps):
    window = new_file_df.iloc[i:i+timesteps][X_cols].values
    pred_sequences.append(window)

X_pred = np.array(pred_sequences)  # shape: (num_windows, timesteps, num_features)


In [ ]:
y_pred = model.predict(X_pred)  # shape: (num_windows, 1)
y_pred = y_pred.flatten()


In [ ]:
from sklearn.metrics import mean_absolute_error

def plot_predictions_with_zoom(y_true, y_pred, y_col='target', offset=0,
                                zoom_range=(3000, 3200), log_scale=True):
    """
    Plots predicted vs. true values with MAE and a zoomed-in subplot.

    Parameters:
    - y_true: Array of ground truth values (will be shifted by offset).
    - y_pred: Array of predicted values.
    - y_col: Name of the target variable (for axis labeling).
    - offset: Integer offset if prediction starts after timesteps.
    - zoom_range: Tuple of (x_min, x_max) for the zoomed-in view.
    - log_scale: Whether to apply symlog scale to the y-axis.
    """
    if offset:
        y_true = y_true[offset:]
    
    mae = mean_absolute_error(y_true, y_pred)

    fig, axs = plt.subplots(2, 1, figsize=(12, 7), sharey=True)

    # --- Full Plot ---
    axs[0].plot(y_true, label=f'True {y_col}', linewidth=2)
    axs[0].plot(y_pred, label=f'Predicted {y_col}', linewidth=2, linestyle='--')
    axs[0].set_title(f'Predicted vs. True {y_col.capitalize()} (Full)\nMAE = {mae:.4f} $E_h$', fontsize=14)
    axs[0].set_ylabel(f'{y_col.capitalize()}')
    axs[0].legend()
    axs[0].grid(True, linestyle='--', linewidth=0.5)
    if log_scale:
        axs[0].set_yscale("symlog")

    # --- Zoomed-In Plot ---
    axs[1].plot(y_true, label=f'True {y_col}', linewidth=2)
    axs[1].plot(y_pred, label=f'Predicted {y_col}', linewidth=2, linestyle='--')
    axs[1].set_title(f'Zoomed In: Steps {zoom_range[0]}–{zoom_range[1]}', fontsize=13)
    axs[1].set_xlabel('Time Step')
    axs[1].set_ylabel(f'{y_col.capitalize()}')
    axs[1].set_xlim(zoom_range)
    axs[1].legend()
    axs[1].grid(True, linestyle='--', linewidth=0.5)
    if log_scale:
        axs[1].set_yscale("symlog")

    plt.tight_layout()
    plt.show()



In [ ]:
plot_predictions_with_zoom(
    y_true=new_file_df[y_col].values,
    y_pred=y_pred,
    y_col=y_col,
    offset=timesteps,
    zoom_range=(3000, 3200),
    log_scale=False
)
